In [116]:
import re
import requests
import geocoder
import time
import webbrowser
import sys


In [68]:
items = []

print('')

while True:
    item = input("welke gerechten heb je al/ heb je die je wilt gebruiken. (type stop om te stoppen): ")

    if item == "stop":
        break

    items.append(item)

In [69]:
g = geocoder.ip("me")
regio = g.city
lat = g.latlng[0]
lon = g.latlng[1]

In [104]:
print(lat)
print(lon)

52.0767
4.2986


In [79]:
prompts = [
    f"""
Genereer EXACT 5 verschillende gerechten op basis van deze ingrediënten:

{items}

BELANGRIJKE REGELS:
- Geef EXACT 5 regels terug.
- Geen uitleg voor of na de regels.
- Geen markdown.
- Geen opsommingstekens.
- Elke regel moet EXACT dit formaat hebben:

gerecht_nr | gerecht | ingredienten | duur | nieuw ingredient = (ingredient)

VOORBEELD:
1 | Pasta met kip | kip, pasta, ui | 25 min | nieuw ingredient = (parmezaanse kaas)

EXTRA REGELS:
- Gebruik zoveel mogelijk ingrediënten uit de opgegeven lijst.
- Maximaal 1 nieuw ingrediënt per gerecht.
- Bereidingstijd maximaal 30 minuten.
- Het nieuwe ingrediënt MOET tussen haakjes staan.
- De tekst 'nieuw ingredient = (...)' moet altijd het laatste onderdeel van de regel zijn.
- Gebruik het pipe-teken '|' uitsluitend als scheidingsteken.
- Gebruik geen pipe-tekens in gerechtnamen of ingrediënten.
- Nummer de gerechten van 1 t/m 5.

Geef alleen de 5 regels terug.
"""
]

In [80]:
for prompt in prompts:
    start_time = time.time()

In [81]:
data = {
    "model": "nvidia/nemotron-3-nano-4b",
    "messages": [
        {"role": "user", "content": prompt}
    ],
    "temperature": 0.7
}


In [110]:
url = 'http://127.0.0.1:1234/v1/chat/completions'

def lmbot():
    try:
        response = requests.post(url, json=data)
        response.raise_for_status()

        end_time = time.time()

        result = response.json()

        answer = result["choices"][0]["message"]["content"]
        model_name = result.get("model", "onbekend")
        response_length = len(answer)

    except requests.exceptions.RequestException as e:
        print(f"Fout bij API-aanroep: {e}")


    for answe in answer.splitlines():
        if "nieuw ingredient" in answe.lower():
            ingredient = answe.split("=", 1)[1].strip()

            url2 = f"https://google.com/search?q=supermarkt+voor+{ingredient}+-ai"
            answe += f"|{url2}"

        print(answe)

    print("\nMetadata:")
    print(f"Modelnaam: {model_name}")
    print(f"Lengte response: {response_length} karakters")
    print(f"Tijd: {end_time - start_time:.2f} seconden")
    print("-" * 50)

In [111]:
while True:
    lmbot()

    check = input("Zijn dit ingrediënten die je wilt? (Ja/Nee) ")

    if check.lower() == "ja":
        break


1 | Kip met sla en aardappels | kip, sla, aardappels | 20 min | nieuw ingredient = (zalm)  |https://google.com/search?q=supermarkt+voor+(zalm)+-ai
2 | Zalm en broccoli | zalm, broccoli | 15 min | nieuw ingredient = (bonen)  |https://google.com/search?q=supermarkt+voor+(bonen)+-ai
3 | Aardappels, sla en bonen | aardappels, sla, bonen | 20 min | nieuw ingredient = (kip)  |https://google.com/search?q=supermarkt+voor+(kip)+-ai
4 | Kip met broccoli | kip, broccoli | 18 min | nieuw ingredient = (sla)  |https://google.com/search?q=supermarkt+voor+(sla)+-ai
5 | Zalm en aardappels | zalm, aardappels | 22 min | nieuw ingredient = (bonen)|https://google.com/search?q=supermarkt+voor+(bonen)+-ai

Metadata:
Modelnaam: nvidia/nemotron-3-nano-4b
Lengte response: 417 karakters
Tijd: 1719.96 seconden
--------------------------------------------------


In [120]:
destination = input('wat is de supermarkt waar je heen wilt( vul supermarkt in of niks)')

In [121]:
if destination in ("niks", ""):
    sys.exit()

else:
    url2 = (
        f"https://www.google.com/maps/dir/?api=1"
        f"&origin={lat},{lon}"
        f"&destination={destination}"
        f"&travelmode=driving"
    )

    webbrowser.open(url2)